## Aula 09 - 27/10/2026


Neste notebook, usaremos o dataset Titanic para relacionar matemática discreta, probabilidade e estatística descritiva com operações vetorizadas do Pandas.

Ao final, você deverá ser capaz de:

- aplicar os princípios aditivo e multiplicativo a variáveis categóricas;
- calcular frequências conjuntas e probabilidades condicionais;
- comparar média, mediana e desvio padrão por subgrupo;
- identificar diferenças entre o espaço combinatório teórico e os dados observados.

**Fonte dos dados:** Data Science Dojo, dataset Titanic, com 891 registros de passageiros. As análises deste notebook descrevem essa amostra; não representam uma inferência causal sobre sobrevivência.

#### Recursos do Pandas utilizados na próxima célula

- `import pandas as pd`: importa a biblioteca Pandas e define `pd` como seu nome de referência no notebook.
- `pd.set_option()`: altera configurações de exibição. Neste notebook, limita a quantidade de colunas apresentadas e define duas casas decimais na visualização, sem arredondar os valores armazenados.
- `pd.read_csv()`: lê um arquivo CSV e retorna um `DataFrame`. Como recebe uma URL, os dados são carregados diretamente do endereço informado.
- `df.shape`: atributo que informa as dimensões do `DataFrame` no formato `(número_de_linhas, número_de_colunas)`.
- `df.head()`: retorna as primeiras linhas do `DataFrame`; por padrão, apresenta cinco registros para inspeção inicial.

In [40]:
import pandas as pd
from itertools import product

pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 2)

url = (
    "https://raw.githubusercontent.com/"
    "datasciencedojo/datasets/master/titanic.csv"
)

df = pd.read_csv(url)

# Se a URL principal estiver indisponível, use temporariamente:
# url = "https://cdn.jsdelivr.net/gh/datasciencedojo/datasets@master/titanic.csv"
# df = pd.read_csv(url)

print(f"Dimensões do DataFrame: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head()

Dimensões do DataFrame: 891 linhas × 12 colunas


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.28,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.92,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.10,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,NaN,S


### 1. Inspeção inicial

Antes de contar ou calcular probabilidades, precisamos verificar:

1. quais colunas estão disponíveis;
2. quais tipos de dados foram inferidos;
3. onde existem valores ausentes;
4. se a quantidade de registros corresponde ao espaço amostral esperado.

Nesta aula, cada linha representa um passageiro. Assim, o espaço amostral completo do dataset possui cardinalidade $|\Omega| = 891$.

#### Recursos do Pandas utilizados na próxima célula

- `df.info()`: apresenta um resumo da estrutura do `DataFrame`, incluindo quantidade de registros, nomes das colunas, tipos de dados e quantidade de valores não nulos.
- `df.isna()`: identifica valores ausentes, produzindo uma estrutura booleana em que `True` indica um valor nulo.
- `.sum()`: soma os valores de cada coluna. Quando aplicado a resultados booleanos, cada `True` conta como 1; por isso, `isna().sum()` conta os valores ausentes por coluna.
- `.rename("valores_ausentes")`: atribui um nome descritivo à `Series` resultante.
- `.to_frame()`: converte uma `Series` em um `DataFrame`, facilitando sua apresentação em formato de tabela.

In [41]:
display(df.info())

colunas_aula = ["Pclass", "Sex", "Embarked", "Age", "Fare", "Survived"]

display(
    df[colunas_aula]
    .isna()
    .sum()
    .rename("valores_ausentes")
    .to_frame()
)

print("Verificações iniciais concluídas.")

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


None

,valores_ausentes
Pclass,0
Sex,0
Embarked,2
Age,177
Fare,0
Survived,0


Verificações iniciais concluídas.


### 2. Princípio Aditivo: partições e contagens

Se os conjuntos $A_1, A_2, \ldots, A_k$ são disjuntos dois a dois, então:

$$
\left| A_1 \cup A_2 \cup \cdots \cup A_k \right|
=
|A_1| + |A_2| + \cdots + |A_k|.
$$

Em uma coluna sem valores nulos, cada categoria define um subconjunto disjunto dos registros. Por isso, as contagens de `Pclass` recompõem o total de passageiros.

O método `value_counts()` conta quantas linhas pertencem a cada categoria. Por padrão, ele ignora valores ausentes (`dropna=True`).

#### Recursos do Pandas utilizados na próxima célula

- `Series.value_counts()`: conta a frequência de cada valor distinto de uma coluna. Por padrão, não inclui valores ausentes no resultado.
- `.sort_index()`: ordena o resultado pelos rótulos do índice. Em `Pclass`, isso organiza as contagens na sequência 1ª, 2ª e 3ª classes, em vez de ordená-las pela frequência.
- `.rename("contagem")`: nomeia a `Series` de frequências.
- `.to_frame()`: transforma a contagem em uma tabela com uma coluna.
- `Series.sum()`: soma as frequências das categorias, permitindo verificar se a partição recompõe o total de registros.

In [42]:
particao_classe = df["Pclass"].value_counts().sort_index()

print("Passageiros por classe:")
display(particao_classe.rename("contagem").to_frame())

total_recomposto = int(particao_classe.sum())

print(f"Soma das partições: {total_recomposto}")
print(f"Cardinalidade de Ω: {len(df)}")

Passageiros por classe:


,contagem
Pclass,
1,216
2,184
3,491


Soma das partições: 891
Cardinalidade de Ω: 891


#### Atenção aos valores nulos

A soma das contagens só recompõe $|\Omega|$ quando a coluna não possui valores ausentes, ou quando incluímos explicitamente a categoria de valores nulos.

`Embarked` possui dois valores ausentes. Compare as duas chamadas abaixo e observe quando a soma deixa de ser igual a 891.

#### Recursos do Pandas utilizados na comparação

- `value_counts(dropna=True)`: comportamento padrão, que exclui valores ausentes das contagens.
- `value_counts(dropna=False)`: inclui os valores ausentes como uma categoria adicional, identificada por `NaN`.
- `.sum()`: totaliza as frequências. Sem os nulos, a soma será 889; incluindo os dois registros ausentes de `Embarked`, ela voltará a 891.
- `.rename()` e `.to_frame()`: apenas nomeiam e formatam o resultado para apresentação, sem alterar as contagens.

In [43]:
sem_nulos = df["Embarked"].value_counts()              # dropna=True
com_nulos = df["Embarked"].value_counts(dropna=False)  # inclui NaN

print("Sem incluir valores nulos:")
display(sem_nulos.rename("contagem").to_frame())
print(f"Soma: {sem_nulos.sum()}")

print("Incluindo valores nulos:")
display(com_nulos.rename("contagem").to_frame())
print(f"Soma: {com_nulos.sum()}")

Sem incluir valores nulos:


,contagem
Embarked,
S,644
C,168
Q,77


Soma: 889
Incluindo valores nulos:


,contagem
Embarked,
S,644
C,168
Q,77
NaN,2


Soma: 891


### 3. Princípio Multiplicativo: espaço combinatório

Se uma configuração é formada por $k$ escolhas independentes, com $n_1, n_2, \ldots, n_k$ possibilidades em cada etapa, o número de configurações teóricas é:

$$
n_1 \times n_2 \times \cdots \times n_k.
$$

No dataset, usaremos três dimensões:

- `Pclass`: 1ª, 2ª ou 3ª classe;
- `Sex`: `female` ou `male`;
- `Embarked`: Southampton (`S`), Cherbourg (`C`) ou Queenstown (`Q`).

O produto dos domínios fornece o **teto combinatório teórico**. O `groupby()` revela quais dessas combinações realmente aparecem nos dados.

### Recursos do Pandas utilizados na próxima célula

- `df[dimensoes_3d]`: seleciona simultaneamente as colunas `Pclass`, `Sex` e `Embarked`.
- `DataFrame.nunique()`: conta os valores distintos de cada coluna. Por padrão, valores ausentes não entram nessa contagem. O resultado fornece a cardinalidade observada de cada domínio.
- `Series.prod()`: multiplica as cardinalidades das três dimensões, calculando o número teórico de perfis pelo Princípio Multiplicativo.
- `df.groupby(dimensoes_3d)`: agrupa os registros que possuem a mesma combinação de classe, sexo e porto de embarque.
- `.size()`: conta quantos registros existem em cada grupo, retornando uma `Series` cujo índice identifica a combinação de categorias.
- `.rename("passageiros")`: atribui um nome à contagem produzida por `size()`.
- `.sort_index()`: ordena as combinações pelos valores das dimensões, facilitando a leitura.
- `.to_frame()`: apresenta os perfis e suas frequências como um `DataFrame`.

A quantidade de perfis observados é obtida com `len(perfis_3d)`: cada entrada da `Series` corresponde a uma combinação efetivamente encontrada nos dados.

In [44]:
dimensoes_3d = ["Pclass", "Sex", "Embarked"]

cardinalidades = df[dimensoes_3d].nunique()

print("Cardinalidade de cada domínio:")
display(cardinalidades.rename("n_i").to_frame())

n_teoricos_3d = int(cardinalidades.prod())

perfis_3d = (
    df.groupby(dimensoes_3d)
    .size()
    .rename("passageiros")
    .sort_index()
)

n_observados_3d = len(perfis_3d)
n_lacunas_3d = n_teoricos_3d - n_observados_3d

print(f"Perfis teóricos: {n_teoricos_3d}")
print(f"Perfis observados: {n_observados_3d}")
print(f"Lacunas: {n_lacunas_3d}")

display(perfis_3d.to_frame())

Cardinalidade de cada domínio:


,n_i
Pclass,3
Sex,2
Embarked,3


Perfis teóricos: 18
Perfis observados: 18
Lacunas: 0


passageiros
Pclass Sex    Embarked             
1      female C                  43
              Q                   1
              S                  48
       male   C                  42
              Q                   1
              S                  79
2      female C                   7
              Q                   2
              S                  67
       male   C                  10
              Q                   1
              S                  97
3      female C                  23
              Q                  33
              S                  88
       male   C                  43
              Q                  39
              S                 265

### 4. Tabelas de contingência: frequências conjuntas

Uma tabela de contingência organiza simultaneamente duas variáveis categóricas. Cada célula interna representa a cardinalidade de uma interseção, como:

$$
\left|\mathrm{Pclass}_i \cap \mathrm{Survived}_j\right|.
$$

Com `margins=True`, o Pandas acrescenta totais de linhas e colunas. Esses totais são aplicações diretas do Princípio Aditivo.

#### Recursos do Pandas utilizados na próxima célula

- `pd.crosstab()`: constrói uma tabela de contingência cruzando duas ou mais variáveis. Neste caso, as linhas representam `Pclass`, as colunas representam `Survived`, e as células internas contêm frequências absolutas.
- `margins=True`: acrescenta os totais marginais de linhas e colunas.
- `margins_name="Total"`: define o rótulo apresentado para esses totais.
- `.loc[...]`: indexador do Pandas usado para selecionar dados por rótulos. Por exemplo, `tab_abs.loc[1, 1]` acessa a célula correspondente à 1ª classe e aos sobreviventes.
- `.sum()`: soma frequências selecionadas, permitindo conferir se os subtotais recompõem o total geral.

In [45]:
tab_abs = pd.crosstab(
    df["Pclass"],
    df["Survived"],
    margins=True,
    margins_name="Total"
)

display(tab_abs)

Survived,0,1,Total
Pclass,,,
1,80,136,216
2,97,87,184
3,372,119,491
Total,549,342,891


### 5. Probabilidade marginal e condicional

A probabilidade marginal usa todo o espaço amostral:

$$
P(S)=\frac{|S|}{|\Omega|}
=\frac{342}{891}
\approx 0{,}384.
$$

Na probabilidade condicional, restringimos o espaço amostral ao evento conhecido $B$:

$$
P(A\mid B)
=\frac{|A\cap B|}{|B|}
=\frac{P(A\cap B)}{P(B)}.
$$

Com `normalize="index"`, cada linha da tabela é dividida pelo total da própria linha. O resultado fornece $P(\mathrm{Survived}\mid\mathrm{Pclass})$ para todas as classes em uma única operação.

#### Recursos do Pandas utilizados na próxima célula

- `pd.crosstab(normalize="index")`: divide cada célula pelo total da linha correspondente. Assim, cada linha passa a somar 1 e representa a distribuição condicional de `Survived` dentro de uma classe.
- `.round(1)`: arredonda os valores para uma casa decimal, melhorando a apresentação dos percentuais.
- `Series.mean()`: calcula a média aritmética. Como `Survived` contém apenas 0 e 1, sua média equivale à proporção de sobreviventes.

In [46]:
prob_cond = pd.crosstab(
    df["Pclass"],
    df["Survived"],
    normalize="index"
) * 100

print("Probabilidades condicionais por classe (%):")
display(prob_cond.round(1))

print(f"Sobrevivência marginal: {df['Survived'].mean() * 100:.1f}%")

Probabilidades condicionais por classe (%):


Survived,0,1
Pclass,,
1,37.0,63.0
2,52.7,47.3
3,75.8,24.2


Sobrevivência marginal: 38.4%
